# Blackjack environment: split, double and a finite shoe

Gymnasium's `Blackjack-v1` gives the agent only two actions: **stick (0)** and **hit (1)**.
This notebook prepares the game environment that the other notebooks build on. It adds
**double (2)**, **split (3)** and a finite shoe.

The code lives in the `blackjack_rl` package (`src/blackjack_rl/`), so every notebook and the
pygame table import the same rules. This notebook walks through it and checks it. The unit tests
are in `tests/` (`uv run pytest`).

| Module | What it holds |
|---|---|
| `rules` | `State`, `Rules`, and the game as pure functions: `actions`, `result`, `utility`, ... |
| `shoe` | `new_shoe`, `needs_shuffle` |
| `env` | `BlackjackSplitDoubleEnv`, registered as `BlackjackSplitDouble-v0` |
| `policies` | baseline policies: `random_policy`, `stick_on_17`, `double_and_split`, `basic_strategy` |
| `play` | `play_episode`, `evaluate` |

The rules follow AIMA: **pure functions over an immutable state**. `actions(s)`,
`result(s, a, draw)`, `is_terminal(s)` and `utility(s)` are the whole game. The Gymnasium `Env`
is a thin shell that feeds them cards.

In [ ]:
import inspect

import gymnasium as gym
from IPython.display import Code, display

from blackjack_rl import ENV_ID, PYGAME_TABLE_RULES, Rules
from blackjack_rl.play import evaluate, play_episode
from blackjack_rl.policies import basic_strategy, double_and_split, random_policy, stick_on_17
from blackjack_rl.rules import DOUBLE, SPLIT, STICK, State, actions, dealer_play, result, utility


def show(obj) -> None:
    """Display an object's source from the package."""
    display(Code(inspect.getsource(obj), language="python"))

## House rules

A frozen `Rules` value describes the table. Every rule function takes one, so changing the
table means passing a different value, never editing code. `PYGAME_TABLE_RULES` matches the
pygame game in `blackjack_pygame/`. Train with it if you want an agent to play that table.

In [ ]:
show(Rules)
print("pygame table:", PYGAME_TABLE_RULES)

## State and rules

One immutable `State` holds everything. A split turns one hand into two, so each hand has its
own bet and *done* flag. The active hand is the first one that isn't done.

In [ ]:
show(State)
show(actions)
show(result)

`draw` is a zero-argument function that returns a card, so the rules never touch randomness.
That means a fixed deck can drive them. Here we split 8s against a dealer showing 10 with 7
underneath:

In [ ]:
def deck_of(*cards: int):
    return iter(cards).__next__


s = State(hands=((8, 8),), bets=(1.0,), done=(False,), dealer=(10, 7))
print(actions(s))                              # stick, hit, double, split
s = result(s, SPLIT, deck_of(3, 10))           # hands (8,3) and (8,10)
s = result(s, DOUBLE, deck_of(10))             # 8+3+10 = 21 for two bets
s = result(s, STICK, deck_of())                # 18 stands
s = dealer_play(s, deck_of())                  # dealer already has 17
print(s, "->", utility(s))                     # +2 and +1

## The environment

`import blackjack_rl` registers `BlackjackSplitDouble-v0`. Keyword arguments to `gym.make`
override the default rules. The observation is `Blackjack-v1`'s
`(player_sum, dealer_card, usable_ace)` plus `(can_double, can_split)`. Rewards are in units of
the original bet, and the round is paid out on the step that ends it.

Each step's `info` carries everything the later notebooks need:

| key | use |
|---|---|
| `action_mask` | legal actions: `action_space.sample(mask=...)`, or masking Q-values in a DQN |
| `revealed` | cards that became visible this step. The hole card only appears when the round ends |
| `shuffled` | a fresh shoe was opened, so reset the count |
| `cards_left` | cards left in the shoe, for a true count |
| `state` | the full `State`, **hole card included**. Use it for debugging, never as an agent input |

Illegal actions raise an error rather than being ignored.

In [ ]:
env = gym.make(ENV_ID, natural=True, num_decks=6)
for seed in range(3):
    play_episode(env, random_policy, seed=seed, verbose=True)
    print()

## The shoe

`num_decks=None` (the default) is `Blackjack-v1`'s infinite deck. With `num_decks=n`, cards come
from a shuffled shoe of `n` decks. The shoe is reshuffled between rounds once `penetration` of it
has been dealt. A finite shoe is just a different `draw`, so the rule functions don't change.

In [ ]:
env = gym.make(ENV_ID, num_decks=1, penetration=0.75)
print([env.reset(seed=0 if i == 0 else None)[1]["cards_left"] for i in range(12)])

## Baselines

`evaluate` returns the mean reward per round with its standard error. These are the numbers a
learned agent has to beat. Basic strategy should do best of the three.

In [ ]:
env = gym.make(ENV_ID, natural=True, num_decks=6)
for policy in (stick_on_17, double_and_split, basic_strategy):
    print(f"{policy.__name__:>17}: {evaluate(env, policy)}")

Shoe size matters a little: fewer decks favour the player slightly. The effect is close to the
standard error at this sample size, so raise `n` to see it clearly.

In [ ]:
for num_decks in (None, 8, 6, 2, 1):
    label = "infinite" if num_decks is None else f"{num_decks}-deck"
    env = gym.make(ENV_ID, natural=True, num_decks=num_decks)
    print(f"{label:>9}: {evaluate(env, basic_strategy)}")

## Prep: DQN notebook

A DQN wants a flat vector. Gymnasium's `FlattenObservation` one-hot encodes the five discrete
fields into a length-49 vector:

In [ ]:
from gymnasium.wrappers import FlattenObservation

env = FlattenObservation(gym.make(ENV_ID, num_decks=6))
obs, info = env.reset(seed=0)
print(obs.shape, info["action_mask"])

Points for the DQN notebook:
- **Masking:** set illegal Q-values to `-inf` with `info["action_mask"]`, both when choosing an
  action and in the target `max`. Use the mask from the *next* state for the target.
- **Sparse reward:** moving between split hands gives reward 0, and the round is paid out at
  the end. With `gamma = 1` the return is just the round's payoff.
- **Matching the table:** train with `gym.make(ENV_ID, rules=PYGAME_TABLE_RULES)` if the agent
  will play the pygame table. Pass `num_decks=` too if the table uses more than one deck.
- **Dependencies:** PyTorch isn't installed yet. Add it from that notebook with `uv add torch`.

## Watching an agent play the pygame table

`blackjack_pygame/game_env.py` wraps the pygame table as a Gymnasium env with the same
observation, actions and `info`. Any policy or wrapper from these notebooks drives it unchanged,
with a pause between moves so you can follow along. `autoplay.py` runs it:

```sh
cd blackjack_pygame
uv run python autoplay.py --policy blackjack_rl.policies:basic_strategy --delay 1.0 --decks 6
```

In Docker, the agent's game shows at the usual noVNC URL:

```sh
AGENT_POLICY=blackjack_rl.policies:basic_strategy AGENT_DECKS=6 docker compose up -d blackjack
```

`--policy` takes any `module:name` callable `(obs, info) -> action`, and `--wrapper` any
`env -> env` callable. Notebooks can't be imported, so when an agent is ready, move its policy
(and any wrapper it was trained with) into `src/blackjack_rl/`. A trained DQN becomes a
small module that loads its weights and exposes a policy.